<a href="https://colab.research.google.com/github/leideng/AI-primer/blob/main/dnn/dnn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# ==============================================================
# Define the Deep Neural Network (DNN)
# ==============================================================
class DNN(nn.Module):
    def __init__(self, input_size=5, hidden_sizes=[64, 128, 64], output_size=2, dropout=0.2):
        super(DNN, self).__init__()

        layers = []
        prev_size = input_size

        # Build hidden layers
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = h_size

        # Output layer
        layers.append(nn.Linear(prev_size, output_size))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# ==============================================================
# Generate synthetic data
# ==============================================================
def generate_data(n_samples=1000, input_size=5, num_classes=2):
    X = torch.randn(n_samples, input_size)
    y = torch.randint(0, num_classes, (n_samples,))
    return X, y


# ==============================================================
# Training function
# ==============================================================
def train(model, dataloader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # Forward pass
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track metrics
            total_loss += loss.item()
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

        avg_loss = total_loss / len(dataloader)
        accuracy = 100.0 * correct / total
        print(f"Epoch [{epoch+1:02d}/{epochs}]  Loss: {avg_loss:.4f}  Accuracy: {accuracy:.2f}%")


# ==============================================================
# Evaluation function
# ==============================================================
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = 100.0 * correct / total
    print(f"\nTest Accuracy: {accuracy:.2f}%")
    return accuracy


# ==============================================================
# Main
# ==============================================================
def main():
    # Hyperparameters
    INPUT_SIZE = 5
    HIDDEN_SIZES = [64, 128, 128, 128, 128, 64]
    OUTPUT_SIZE = 2
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-3
    EPOCHS = 20
    DROPOUT = 0.2

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # Data
    X_train, y_train = generate_data(n_samples=800, input_size=INPUT_SIZE, num_classes=OUTPUT_SIZE)
    X_test, y_test = generate_data(n_samples=200, input_size=INPUT_SIZE, num_classes=OUTPUT_SIZE)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model = DNN(
        input_size=INPUT_SIZE,
        hidden_sizes=HIDDEN_SIZES,
        output_size=OUTPUT_SIZE,
        dropout=DROPOUT
    ).to(device)

    print(model)
    print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}\n")

    # Loss & Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Train
    train(model, train_loader, criterion, optimizer, device, epochs=EPOCHS)

    # Evaluate
    evaluate(model, test_loader, device)

    # Save model
    torch.save(model.state_dict(), "dnn_model.pth")
    print("\nModel saved to dnn_model.pth")


if __name__ == "__main__":
    main()

Using device: cuda

DNN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=128, out_features=128, bias=True)
    (13): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=128, out_features=128, bias=True)
    (17): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine